## Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import skew, kurtosis
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_style('whitegrid')

print('✓ Bibliothèques importées avec succès')

## PARTIE A - Chargement et préparation des données

In [ ]:
# Charger les données
df = pd.read_csv('../data/Base_Maladie_Cardiaque.csv')

print('=' * 70)
print('PARTIE A — Chargement et préparation des données'.center(70))
print('=' * 70)

print(f'\n✓ Données chargées avec succès')
print(f'  • Nombre de lignes (patients)   : {df.shape[0]}')
print(f'  • Nombre de colonnes (variables): {df.shape[1]}')

print(f'\n  Colonnes : {list(df.columns)}')

print(f'\nAperçu des 5 premières lignes :')
df.head()

In [ ]:
# Conversion des variables catégorielles
categorical_vars = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'target']
df[categorical_vars] = df[categorical_vars].apply(lambda x: x.astype('category'))

print('✓ Variables catégorielles converties')
print('\nTypes de données :')
print(df.dtypes)

# Vérification des valeurs manquantes
missing = df.isnull().sum()
print(f'\n✓ Aucune valeur manquante détectée' if missing.sum() == 0 else f'\nValeurs manquantes :\n{missing[missing > 0]}')

## PARTIE B - Analyse d'une variable quantitative : age

In [ ]:
# Calcul des statistiques descriptives pour l'âge
age = df['age']

print('\n' + '=' * 70)
print('PARTIE B — Variable quantitative : age'.center(70))
print('=' * 70)

# Résumé automatique
print('\n--- Résumé automatique (describe) ---')
print(age.describe().round(2))

# Calcul des indicateurs
moyenne    = age.mean()
mediane    = age.median()
ecart_type = age.std()
variance   = age.var()
min_age    = age.min()
max_age    = age.max()
q1         = age.quantile(0.25)
q3         = age.quantile(0.75)
iqr        = q3 - q1
asymetrie  = skew(age)
aplatiss   = kurtosis(age)

In [ ]:
# Tableau récapitulatif des indicateurs
print('\n--- Tableau récapitulatif des indicateurs ---')
print(f'\n{"Indicateur":<20} {"Valeur":>10}   Interprétation')
print('-' * 80)

indicateurs = [
    ('Moyenne', moyenne, 'Âge moyen des patients'),
    ('Médiane', mediane, 'La moitié des patients ont moins que cet âge'),
    ('Écart-type', ecart_type, 'Dispersion moyenne autour de la moyenne'),
    ('Variance', variance, 'Dispersion au carré'),
    ('Minimum', min_age, 'Patient le plus jeune'),
    ('Maximum', max_age, 'Patient le plus âgé'),
    ('Q1', q1, '25% des patients ont moins que cet âge'),
    ('Q3', q3, '75% des patients ont moins que cet âge'),
    ('IQR', iqr, 'Étendue centrale (Q3 - Q1)'),
    ('Asymétrie', asymetrie, 'Skewness : 0=symétrique, >0 queue droite, <0 queue gauche'),
    ('Aplatissement', aplatiss, 'Comparé à une loi normale (référence = 0)'),
]

for nom, val, interp in indicateurs:
    print(f'{nom:<20} {val:>10.2f}   {interp}')

In [ ]:
# Visualisations : 4 graphiques
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analyse univariée — Variable : age', fontsize=14, fontweight='bold')

# Histogramme
sns.histplot(df['age'], bins=25, kde=False, color='steelblue', ax=axes[0, 0], edgecolor='white')
axes[0, 0].axvline(moyenne, color='red', linestyle='--', linewidth=1.5, label=f'Moyenne = {moyenne:.1f}')
axes[0, 0].axvline(mediane, color='orange', linestyle='--', linewidth=1.5, label=f'Médiane = {mediane:.1f}')
axes[0, 0].set_title('Histogramme')
axes[0, 0].set_xlabel('Âge (années)')
axes[0, 0].set_ylabel('Fréquence')
axes[0, 0].legend()

# Boxplot
sns.boxplot(x=df['age'], color='lightgreen', ax=axes[0, 1], flierprops=dict(marker='o', markerfacecolor='red', markersize=5))
axes[0, 1].set_title('Boxplot')
axes[0, 1].set_xlabel('Âge (années)')

# KDE
sns.kdeplot(df['age'], fill=True, color='darkorange', alpha=0.6, ax=axes[1, 0])
axes[1, 0].axvline(moyenne, color='red', linestyle='--', linewidth=1.5, label=f'Moyenne = {moyenne:.1f}')
axes[1, 0].set_title('Courbe de densité (KDE)')
axes[1, 0].set_xlabel('Âge (années)')
axes[1, 0].set_ylabel('Densité')
axes[1, 0].legend()

# QQ Plot
(osm, osr), (slope_val, intercept, r) = stats.probplot(df['age'], dist='norm')
axes[1, 1].plot(osm, osr, 'o', color='steelblue', markersize=4, alpha=0.7)
axes[1, 1].plot(osm, slope_val * osm + intercept, 'r-', linewidth=1.5)
axes[1, 1].set_title(f'QQ Plot (r = {r:.4f})')
axes[1, 1].set_xlabel('Quantiles théoriques')
axes[1, 1].set_ylabel('Quantiles observés')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n✓ Graphiques affichés avec succès')

## PARTIE C - Analyse d'une variable qualitative : sex

In [ ]:
print('\n' + '=' * 70)
print('PARTIE C — Variable qualitative : sex'.center(70))
print('=' * 70)

# Fréquences
freq_abs = df['sex'].value_counts()
freq_prop = df['sex'].value_counts(normalize=True) * 100
mode_sex = df['sex'].mode()[0]

print('\n--- Fréquences absolues et relatives ---')
print(f'\n{"Catégorie":<20} {"Fréquence":>10} {"Proportion":>10}')
print('-' * 45)
for cat in freq_abs.index:
    label = 'Femme (0)' if cat == 0 else 'Homme (1)'
    print(f'{label:<20} {freq_abs[cat]:>10} {freq_prop[cat]:>9.1f}%')

print(f'\nMode (catégorie dominante) : {mode_sex} (Homme)' if mode_sex == 1 else f'\nMode (catégorie dominante) : {mode_sex} (Femme)')
print('→ Déséquilibre important : les hommes sont sur-représentés')

In [ ]:
# Visualisations pour la variable qualitative
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse univariée — Variable : sex', fontsize=14, fontweight='bold')

# Countplot
sns.countplot(x='sex', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Diagramme en barres (Effectifs)')
axes[0].set_xlabel('Sexe')
axes[0].set_ylabel('Nombre de patients')
axes[0].set_xticklabels(['Femme (0)', 'Homme (1)'])
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom')

# Pie chart
values = df['sex'].value_counts()
labels = ['Femme' if i == 0 else 'Homme' for i in values.index]
colors = ['#ff9999', '#66b3ff']
axes[1].pie(values.values, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
axes[1].set_title('Répartition (Proportions)')

plt.tight_layout()
plt.show()

print('\n✓ Graphiques affichés avec succès')

## PARTIE D - Automatisation et tableaux récapitulatifs

In [ ]:
# Tableau récapitulatif pour toutes les variables quantitatives
print('\n' + '=' * 70)
print('PARTIE D — Tableau récapitulatif automatisé'.center(70))
print('=' * 70)

quantitative_vars = df.select_dtypes(include=['number']).columns.tolist()
print(f'\nVariables quantitatives détectées : {quantitative_vars}')

stats_df = pd.DataFrame({
    'N': df[quantitative_vars].count(),
    'Moyenne': df[quantitative_vars].mean().round(2),
    'Std': df[quantitative_vars].std().round(2),
    'Min': df[quantitative_vars].min(),
    'Q1': df[quantitative_vars].quantile(0.25).round(2),
    'Médiane': df[quantitative_vars].quantile(0.50).round(2),
    'Q3': df[quantitative_vars].quantile(0.75).round(2),
    'Max': df[quantitative_vars].max(),
    'IQR': (df[quantitative_vars].quantile(0.75) - df[quantitative_vars].quantile(0.25)).round(2),
    'Skewness': df[quantitative_vars].apply(skew).round(3),
    'Kurtosis': df[quantitative_vars].apply(kurtosis).round(3),
})

print('\n')
print(stats_df)

## PARTIE E - Détection des outliers (méthode IQR)

In [ ]:
print('\n' + '=' * 70)
print('PARTIE E — Détection des outliers (méthode IQR)'.center(70))
print('=' * 70)

results = []

for col in quantitative_vars:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    n_outliers = outliers_mask.sum()
    pct_outliers = (n_outliers / len(df)) * 100
    
    results.append({
        'Variable': col,
        'Borne basse': round(lower_bound, 2),
        'Borne haute': round(upper_bound, 2),
        'Nb outliers': n_outliers,
        '% dataset': round(pct_outliers, 1),
    })

outliers_df = pd.DataFrame(results).set_index('Variable')
print('\n')
print(outliers_df)

## Conclusion et Résumé

In [ ]:
print('\n' + '=' * 70)
print('CONCLUSION GÉNÉRALE DU TP1'.center(70))
print('=' * 70)

conclusion = """
Ce TP1 nous a permis de réaliser une analyse exploratoire univariée
complète du Heart Disease Dataset.

POINTS CLÉS :

1. Préparation des données
   → Distinguer le type Python (int/float) de la nature statistique
     (quantitative/qualitative) est indispensable avant toute analyse.

2. Variable age (quantitative continue)
   → Distribution quasi-normale, centrée autour de 54-55 ans.
   → Faible asymétrie (skewness = 0.12) → tests paramétriques applicables.
   → Quelques outliers aux extrêmes, à surveiller.

3. Variable sex (qualitative nominale)
   → Déséquilibre important : ~68% d'hommes dans le dataset.
   → Ce biais doit être mentionné dans toutes les analyses suivantes.

4. Automatisation (Partie D & E)
   → Les fonctions réduisent le temps d'analyse et limitent les erreurs.
   → Détection automatique des outliers : 9 variables analysées.

PROCHAINES ÉTAPES (TP2) :
   → Étudier les relations ENTRE les variables (analyse bivariée).
   → Comment age, chol, thalach évoluent-ils selon la maladie ?
   → Corrélations et régressions simples.
"""

print(conclusion)